# Read and audit native GAMS results

Run this after the GAMS IDE example. It uses the **original workbook reader and daily operational-audit function**, with its static case view derived from the numerical includes already supplied. It does not need the missing governor case to audit UC schedules, and it does not claim a SODE nadir assessment.

## What is checked
Workbook identity; accepted native status; initial-to-final chronological state; binary transitions; minimum up/down obligations; startup categories; ramping; on/off age recurrence; generation/block bounds; power balance; curtailment; reconstructed post-outage inertia/gain/headroom; RoCoF; frozen affine margins; and cost/EUR conversion. Native GAMS itself retains its full equation/bound/integrality residual checks before it exports a state.

## Missing outputs are not fabricated
Before a native run, this notebook completes with **NOT RUN** for result auditing. That is an intentional report, not a successful native solver test. Any incompatible or incomplete workbook raises an exception rather than being silently accepted. Full campaign SODE assessment still needs the original dynamic case/cache and research archive.

In [1]:
from pathlib import Path
import json, hashlib, re
import numpy as np
import pandas as pd


def find_root():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / 'GAMS_LaPalma' / 'START_LAPALMA.gms').is_file():
            return candidate
    raise FileNotFoundError('Extract the complete repository, then open this notebook inside it.')


def read_inc_table(path, row_dimensions):
    """Read the shipped XLS2GMS numerical tables; never evaluate source code.

    Leading star-comments and blank padding are ignored. Dotted row tuples,
    e.g. i1 . b1, identify row dimensions; the first data line is the header.
    The function intentionally supports this repository's tables, not all GAMS.
    """
    lines = [s.strip() for s in Path(path).read_text(encoding='utf-8-sig').splitlines()
             if s.strip() and not s.lstrip().startswith('*')]
    if not lines:
        raise ValueError(f'Empty table: {path}')
    columns = lines[0].split()
    records, labels = [], []
    for line_number, line in enumerate(lines[1:], start=2):
        parts = [x for x in line.split() if x != '.']
        keys, fields = parts[:row_dimensions], parts[row_dimensions:]
        if len(keys) != row_dimensions or len(fields) != len(columns):
            raise ValueError(f'Malformed table row {line_number} in {path}')
        values = [0.0 if v.lower() == 'eps' else float(v) for v in fields]
        if not np.isfinite(values).all():
            raise ValueError(f'Non-finite table values in {path}')
        labels.append(keys[0] if row_dimensions == 1 else tuple(keys))
        records.append(values)
    index = pd.Index(labels) if row_dimensions == 1 else pd.MultiIndex.from_tuples(labels)
    if not index.is_unique:
        raise ValueError(f'Duplicate row identifiers in {path}')
    return pd.DataFrame(records, index=index, columns=columns)


def load_inputs(root):
    inc = Path(root) / 'GAMS_LaPalma' / 'inc'
    ids = [f'i{i}' for i in range(1, 12)]
    def vector(name, col):
        frame = read_inc_table(inc / name, 1)
        return frame.loc[ids, col].to_numpy(dtype=float)
    blocks = read_inc_table(inc / 'block_max.inc', 1).loc[ids, ['b1','b2','b3']]
    units = pd.DataFrame({'unit': ids, 'pmax_mw': blocks.sum(axis=1).to_numpy(),
        'pmin_mw': vector('aux9.inc','tr4'), 'inertia_s': vector('auxinertia.inc','column1'),
        'mbase': vector('auxmbase.inc','column1'), 'gain_pu': vector('auxk.inc','column1'),
        'no_load_keur_h': vector('aux4.inc','tr4'), 'ramp_up_mw_h': vector('aux5.inc','tr4'),
        'ramp_down_mw_h': vector('aux6.inc','tr4'), 'min_up_h': vector('aux8.inc','tr4'),
        'min_down_h': vector('aux7.inc','tr4'), 'initial_on_age_h': vector('aux3.inc','column1'),
        'initial_off_age_h': vector('aux2.inc','column1'), 'initial_power_mw': vector('aux10.inc','column1')})
    units['initial_commitment'] = (units.initial_on_age_h > 0).astype(int)
    units['energy_mws'] = units.inertia_s * units.mbase
    units['gain_mw_hz'] = units.gain_pu * units.mbase / 50.0
    demand = read_inc_table(inc / 'd_seasons.inc', 2)
    wind = read_inc_table(inc / 'RES_seasons.inc', 2)
    profiles = []
    for season in ['winter','spring','summer','autumn']:
        for day in range(1,8):
            for hour in range(1,25):
                key=(f't{hour}', f'd{day}')
                profiles.append({'season':season, 'day':day, 'hour':hour,
                    'demand_mw':float(demand.loc[key,season]), 'wind_available_mw':float(wind.loc[key,season])})
    return units, pd.DataFrame(profiles), blocks

import math
from zipfile import ZipFile
from xml.etree import ElementTree as ET
ROOT=find_root()
UNITS,PROFILES,BLOCKS=load_inputs(ROOT)
DEP=json.loads((ROOT/'GAMS_LaPalma/LP35_Deployment.json').read_text())
IDS=UNITS.unit.tolist();TIDS=[f't{i}' for i in range(1,25)]
PMAX=UNITS.pmax_mw.to_numpy();PMIN=UNITS.pmin_mw.to_numpy()
EUNIT=UNITS.energy_mws.to_numpy();GAIN=UNITS.gain_mw_hz.to_numpy();F0=50.
inc=ROOT/'GAMS_LaPalma/inc'
# This in-memory operational view is derived exclusively from the shipped .inc tables.
# It is NOT an invented LP_Case.json and contains no guessed governor parameters.
CASE={
 'initial_on_h':UNITS.initial_on_age_h.tolist(),'initial_off_h':UNITS.initial_off_age_h.tolist(),
 'initial_power':UNITS.initial_power_mw.tolist(),'min_up_h':UNITS.min_up_h.tolist(),
 'min_down_h':UNITS.min_down_h.tolist(),'ramp_up_mw_h':UNITS.ramp_up_mw_h.tolist(),
 'ramp_down_mw_h':UNITS.ramp_down_mw_h.tolist(),'no_load_keur_h':UNITS.no_load_keur_h.tolist(),
 'blocks':BLOCKS.to_numpy().tolist(),
 'slopes_keur_per_mwh':read_inc_table(inc/'k.inc',2).loc[[(i,f'b{b}') for i in IDS for b in range(1,4)],'tr4'].to_numpy().reshape(11,3).tolist(),
 'start_lower_h':read_inc_table(inc/'start_up_sl.inc',1).loc[IDS,[f'j{i}' for i in range(1,9)]].to_numpy().tolist(),
 'start_cost_keur':read_inc_table(inc/'start_up_sw.inc',2).loc[[(i,f'j{j}') for i in IDS for j in range(1,9)],'tr4'].to_numpy().reshape(11,8).tolist(),
 'rocof_max_hz_s':2.5,
 'demand':{s:[PROFILES.query('season==@s and day==@d').demand_mw.tolist() for d in range(1,8)] for s in DEP['seasons']},
 'res':{s:[PROFILES.query('season==@s and day==@d').wind_available_mw.tolist() for d in range(1,8)] for s in DEP['seasons']}}


## Original readers and audit
The source functions below are preserved verbatim. Excel is only a reporting interface. It never replaces `state.gdx` as the next-day input. The in-memory `CASE` above contains only UC table quantities and the explicitly coded RoCoF constant; no original case identity is assigned to it.

In [2]:
def excel_values(path, sheet):
    ns={'x':'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
    rid='{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id'
    with ZipFile(path) as z:
        strings=[]
        if 'xl/sharedStrings.xml' in z.namelist():
            strings=[''.join(x.itertext()) for x in ET.fromstring(z.read('xl/sharedStrings.xml')).findall('x:si',ns)]
        rel={x.attrib['Id']:x.attrib['Target'] for x in ET.fromstring(z.read('xl/_rels/workbook.xml.rels'))}
        sheets=ET.fromstring(z.read('xl/workbook.xml')).find('x:sheets',ns)
        target=next((rel[s.attrib[rid]] for s in sheets if s.attrib['name']==sheet),None)
        if target is None:raise KeyError(f'{Path(path).name}: missing sheet {sheet}')
        target=target.lstrip('/') if target.startswith('/') else 'xl/'+target
        rows=[]
        for row in ET.fromstring(z.read(target)).findall('x:sheetData/x:row',ns):
            d={}
            for c in row:
                col=0
                for a in re.match(r'[A-Z]+',c.attrib['r']).group():col=col*26+ord(a)-64
                v=c.find('x:v',ns);typ=c.attrib.get('t')
                if typ=='s':value=strings[int(v.text)] if v is not None else None
                elif typ=='inlineStr':value=''.join(c.find('x:is',ns).itertext())
                elif v is None:value=None
                elif typ in ('str','e'):value=v.text
                else:
                    try:value=float(v.text)
                    except (ValueError,TypeError):value=v.text
                d[col-1]=value
            rows.append(d)
        width=max((max(r,default=-1)+1 for r in rows),default=0)
        return [[r.get(j) for j in range(width)] for r in rows]

def number(value):
    if value is None or value=='':return 0.
    if isinstance(value,str) and value.strip().lower() in ('eps','epsilon'):return 0.
    out=float(value)
    if not math.isfinite(out):raise ValueError(f'Nonfinite native number: {value}')
    return out

def read_matrix(path,sheet,row_ids,col_ids):
    data=excel_values(path,sheet);result=np.zeros((len(row_ids),len(col_ids)))
    headers=next((r for r in data if sum(str(x) in col_ids for x in r)>=1),None)
    if headers is None:raise ValueError(f'{sheet}: no expected column IDs')
    positions={str(x):j for j,x in enumerate(headers) if str(x) in col_ids}
    if set(positions)!=set(col_ids):raise ValueError(f'{sheet}: incomplete column identifiers')
    mapped={str(r[0]):r for r in data if r and str(r[0]) in row_ids}
    if set(mapped)!=set(row_ids):raise ValueError(f'{sheet}: incomplete row identifiers')
    for i,key in enumerate(row_ids):
        for j,col in enumerate(col_ids):result[i,j]=number(mapped[key][positions[col]])
    return result

def read_pairs(path,sheet,expected=None):
    result={}
    for row in excel_values(path,sheet):
        if len(row)>=2 and row[0] is not None and (expected is None or str(row[0]) in expected):
            try:result[str(row[0])]=number(row[1])
            except (ValueError,TypeError):continue
    return result

def read_cube(path,sheet,axis1,axis2,axis3):
    data=excel_values(path,sheet);header=next((r for r in data if sum(str(x) in axis3 for x in r)>=1),None)
    if header is None:raise ValueError(f'{sheet}: missing headers')
    pos={str(x):j for j,x in enumerate(header) if str(x) in axis3}
    if set(pos)!=set(axis3):raise ValueError(f'{sheet}: incomplete third-axis labels')
    out=np.zeros((len(axis1),len(axis2),len(axis3)));seen=set();last=None
    for row in data:
        if not row:continue
        if str(row[0]) in axis1:last=str(row[0])
        if len(row)<2 or str(row[1]) not in axis2 or last is None:continue
        key=(last,str(row[1]));seen.add(key)
        for k,c in enumerate(axis3):out[axis1.index(last),axis2.index(str(row[1])),k]=number(row[pos[c]])
    if len(seen)!=len(axis1)*len(axis2):raise ValueError(f'{sheet}: incomplete row identifiers ({len(seen)})')
    return out

def load_daily(path,run,season,day,dep,previous=None):
    if not Path(path).parent.joinpath('COMPLETE.txt').is_file():raise ValueError('Native day has no COMPLETE marker')
    identity={}
    for row in excel_values(path,'Identity'):
        if len(row)>1 and str(row[0]) in ('campaign','run','season','model'):identity[str(row[0])]=str(row[1])
    expected={'campaign':dep['campaign'],'run':run,'season':season,'model':dep['model_id']}
    if identity!=expected:raise ValueError(f'Wrong workbook identity: {identity}; expected {expected}')
    summ=read_pairs(path,'Summary')
    if summ.get('day')!=day or summ.get('horizon_hours')!=24 or summ.get('accepted')!=1:raise ValueError('Wrong day/horizon or unaccepted solve')
    if summ.get('modelstat') not in (1,8) or summ.get('solvestat') not in (1,3):raise ValueError('No accepted integer native schedule')
    # The original 0.5% target remains reported, but a targeted-recovery COMPLETE
    # schedule is not rejected solely because its recorded search gap is larger.
    initial=read_matrix(path,'Initial',IDS,['commitment','power_mw','on_age_h','off_age_h'])
    final=read_matrix(path,'Final',IDS,['commitment','power_mw','on_age_h','off_age_h'])
    declared=np.c_[np.asarray(CASE['initial_on_h'])>0,CASE['initial_power'],CASE['initial_on_h'],CASE['initial_off_h']]
    expectedinitial=declared if day==1 else previous
    if expectedinitial is None or not np.allclose(initial,expectedinitial,atol=1e-6,rtol=0):raise ValueError('Initial state/previous-day transfer differs')
    power=read_matrix(path,'Dispatch',TIDS,IDS);u0=read_matrix(path,'Commitment',TIDS,IDS)
    starts=read_matrix(path,'Startup',TIDS,IDS);stops=read_matrix(path,'Shutdown',TIDS,IDS)
    onraw=read_matrix(path,'OnAge',TIDS,IDS);offraw=read_matrix(path,'OffAge',TIDS,IDS)
    b=read_cube(path,'Blocks',TIDS,IDS,['b1','b2','b3'])
    sc=read_cube(path,'StartCategories',TIDS,IDS,[f'j{k}' for k in range(1,9)])
    curtp=read_pairs(path,'Curtailment',TIDS);dp=read_pairs(path,'Demand',TIDS);wp=read_pairs(path,'WindAvailable',TIDS)
    if any(len(x)!=24 for x in [curtp,dp,wp]):raise ValueError('Missing hourly demand/wind/curtailment values')
    demand=np.array([dp[t] for t in TIDS]);wind=np.array([wp[t] for t in TIDS]);curt=np.array([curtp[t] for t in TIDS])
    if not np.allclose(demand,CASE['demand'][season][day-1],atol=1e-7,rtol=0) or not np.allclose(wind,CASE['res'][season][day-1],atol=1e-7,rtol=0):raise ValueError('Season/day input profile changed')
    binaryerr=max(float(np.max(abs(x-np.rint(x)))) for x in [u0,starts,stops,sc])
    if binaryerr>1e-5:raise ValueError('Nonbinary commitment/switch/category')
    u=np.rint(u0);sy=np.rint(starts);sz=np.rint(stops);sc=np.rint(sc)
    ageon=[];ageoff=[];on=np.rint(initial[:,2]);off=np.rint(initial[:,3]);uprev=np.rint(initial[:,0]);pprev=initial[:,1]
    violations=[]
    def check(name,err,tol=2e-5):
        err=float(np.max(np.asarray(err)))
        if err>tol:violations.append((name,err,tol))
        return err
    for k in range(24):
        check('binary transition',abs(u[k]-uprev-sy[k]+sz[k]),1e-6)
        check('exclusive switching',sy[k]+sz[k]-1,1e-6)
        check('minimum up obligation on shutdown',np.where(sz[k]>0,np.array(CASE['min_up_h'])-on,0),1e-6)
        check('minimum down obligation on startup',np.where(sy[k]>0,np.array(CASE['min_down_h'])-off,0),1e-6)
        thresholds=np.asarray(CASE['start_lower_h']);cat=(thresholds<=off[:,None]+1e-8).sum(1)-1
        expectedcat=np.zeros((11,8))
        for i in range(11):
            if sy[k,i]>.5:
                if cat[i]<0:violations.append(('no eligible startup category',i,0))
                else:expectedcat[i,cat[i]]=1
        check('startup category',abs(sc[k]-expectedcat),1e-6)
        check('ramp up',power[k]-pprev-np.array(CASE['ramp_up_mw_h']))
        check('ramp down',pprev-power[k]-np.array(CASE['ramp_down_mw_h']))
        on=(on+1)*u[k];off=(off+1)*(1-u[k]);ageon.append(on.copy());ageoff.append(off.copy())
        uprev=u[k];pprev=power[k]
    ageon=np.array(ageon);ageoff=np.array(ageoff)
    check('raw on-age drift',abs(onraw-ageon),1e-3);check('raw off-age drift',abs(offraw-ageoff),1e-3)
    check('rounded on-age agreement',abs(np.rint(onraw)-ageon),0)
    check('rounded off-age agreement',abs(np.rint(offraw)-ageoff),0)
    check('final state',abs(final-np.c_[u[-1],power[-1],ageon[-1],ageoff[-1]]),1e-6)
    check('generation lower',PMIN*u-power);check('generation upper',power-PMAX*u);check('negative generation',-power)
    check('block upper',b-np.asarray(CASE['blocks'])[None,:,:]*u[:,:,None]);check('negative block',-b)
    check('block sum',abs(b.sum(2)-power));balance=check('island power balance',abs(power.sum(1)+wind-curt-demand))
    check('negative curtailment',-curt);check('excess curtailment',curt-wind)
    method=next(r for r in dep['runs'] if r['run']==run);frows=[];native_margin=read_matrix(path,'CutMargin',TIDS,IDS)

    sp_extra={}
    if method['mode']==3:raise ValueError('SP is historical context, not a native LP35 run.')
    if method['mode']==3:
        knots1=np.array([14.,26.,38.,52.]);knots2=np.array([5.,11.,17.,22.]);knotsp=np.array([0.,2.5,5.,7.5,10.,12.5])
        w1=read_cube(path,'SPLambda1',TIDS,IDS,[f'x1r{n}' for n in range(1,5)])
        w2=read_cube(path,'SPLambda2',TIDS,IDS,[f'x2r{n}' for n in range(1,5)])
        wp=read_cube(path,'SPLambdaP',TIDS,IDS,[f'pr{n}' for n in range(1,7)])
        sel1=read_cube(path,'SPSelectors1',TIDS,IDS,[f'x1r{n}' for n in range(1,5)])
        sel2=read_cube(path,'SPSelectors2',TIDS,IDS,[f'x2r{n}' for n in range(1,5)])
        selp=read_cube(path,'SPSelectorsP',TIDS,IDS,[f'pr{n}' for n in range(1,7)])
        for w,sel,target,name in [(w1,sel1,np.ones_like(u),'x1'),(w2,sel2,np.ones_like(u),'x2'),(wp,selp,u,'loss')]:
            check('SP negative weights '+name,-w)
            check('SP weight sum '+name,abs(w.sum(2)-target))
            check('SP binary selectors '+name,abs(sel-np.rint(sel)),1e-5)
            check('SP selected interval count '+name,abs(sel[:,:,1:].sum(2)-target))
            # Preserve the exact source adjacent-interval construction, not the IEEE selector variant.
            allowed=np.zeros_like(w);allowed[:,:,1:]+=sel[:,:,1:];allowed[:,:,:-1]+=sel[:,:,1:]
            check('SP weight/interval linkage '+name,w-allowed)
        z1=w1@knots1;z2=w2@knots2
        Eall=(u@EUNIT)[:,None]-u*EUNIT
        Rall=((PMAX*u-power).sum(1))[:,None]-(PMAX*u-power)
        check('SP inertia coordinate',abs((z1+z2)/.5-Eall))
        check('SP reserve coordinate',abs((z1-z2)/2.-Rall))
        check('SP loss coordinate',abs(wp@knotsp-power))
        check('SP x1 lower',14.-z1);check('SP x1 upper',z1-52.)
        check('SP x2 lower',5.-z2);check('SP x2 upper',z2-21.)
        proxy=(w1@(knots1**2)-w2@(knots2**2))/(.5*2.)
        p2=wp@(knotsp**2);credit=.01*demand[:,None]*6.*power*50.*.25
        proxy_margin=proxy-(50.*6./(4.*3.))*p2+credit
        parent_margin=Eall*Rall-(50.*6./(4.*3.))*power**2+credit
        check('SP enforced proxy',-proxy_margin,1e-4)
        # A parent violation is a reported approximation difference, NOT a reason to discard a source-feasible SP schedule.
        sp_extra={'sp_proxy_product_mw2s':proxy,'sp_parent_product_mw2s':Eall*Rall,
                  'sp_loss_squared_proxy':p2,'sp_proxy_margin_mw2s':proxy_margin,
                  'sp_parent_margin_mw2s':parent_margin,
                  'sp_parent_violation':parent_margin < -1e-4}
    for k in range(24):
        for j in range(11):
            survivor=u[k].copy();survivor[j]=0
            E=float(EUNIT@survivor);G=float(GAIN@survivor);R=float(((PMAX*u[k]-power[k])* (np.arange(11)!=j)).sum());loss=float(power[k,j])
            if method['mode']>0:
                check('RoCoF cut',loss-2*E*CASE['rocof_max_hz_s']/F0)
                check('headroom response condition',loss-R)
            margin=None
            if method['mode']==2:
                c=dep['cuts'][run];margin=c['k0']+c['kL']*loss+c['kE']*E+c['kG']*G+c['kR']*R+c['bigM_by_outage'][j]*(1-u[k,j])
                check('affine nadir cut',-margin,1e-5);check('exported cut margin',abs(native_margin[k,j]-margin),1e-5)
            frows.append(dict(run=run,season=season,day=day,hour=k+1,global_hour=(day-1)*24+k+1,outage_unit=IDS[j],candidate_online=int(u[k,j]),loss_mw=loss,inertia_mws=E,gain_mw_hz=G,reserve_mw=R,cut_margin=margin,**{name:float(arr[k,j]) for name,arr in sp_extra.items()}))
    costs=dict(production_keur=float((b*np.asarray(CASE['slopes_keur_per_mwh'])[None,:,:]).sum()),fixed_keur=float((u@np.asarray(CASE['no_load_keur_h'])).sum()),startup_keur=float((sc*np.asarray(CASE['start_cost_keur'])[None,:,:]).sum()),curtailment_cost_keur=0.)
    for key,val in costs.items():check(key,abs(summ[key]-val),1e-5)
    check('objective reconstruction',abs(summ['objective_keur']-sum(costs.values())),1e-5)
    check('EUR conversion',abs(summ['objective_eur']-1000*summ['objective_keur']),1e-3)
    if violations:raise ValueError(f'{run}/{season}/d{day}: {violations[:8]}')
    return dict(summary=summ,power=power,commitment=u,demand=demand,wind=wind,curtailment=curt,initial=initial,final=final,features=frows,balance_residual_mw=balance)


## Selected chronological chain
Use a fresh native output folder for each independent reproduction. The default selection matches the GAMS example. If day 2 exists but day 1 is missing or unaudited, the chain must not be accepted. The source reads saved commitment, dispatch and on/off ages; it does not reset midnight state.

A native objective obtained with a time limit may differ between machines even with unchanged equations. Retain the reported bound, achieved gap and primal residuals; do not invent an exact objective target from a rounded publication table.

In [3]:
SELECT_RUN='LR_C1'
SELECT_SEASON='winter'
assert SELECT_RUN in [r['run'] for r in DEP['runs']]
assert SELECT_SEASON in DEP['seasons']
previous=None;results=[];summaries=[];records=[]
for day in range(1,8):
    folder=ROOT/'GAMS_LaPalma/Results_35Hz'/DEP['campaign']/SELECT_RUN/SELECT_SEASON/f'd{day}'
    path=folder/f'Results_{SELECT_RUN}_{SELECT_SEASON}_d{day}.xlsx'
    if not path.is_file():
        records.append({'day':day,'status':'NOT RUN','reason':'No native workbook supplied/generated.'})
        previous=None
        continue
    if day>1 and previous is None:
        raise ValueError(f'd{day}: preceding day was not available and audited; state continuity cannot be certified.')
    result=load_daily(path,SELECT_RUN,SELECT_SEASON,day,DEP,previous=previous)
    summary=result['summary']
    for key in ['equation_residual','bound_residual','integer_residual']:
        if key not in summary or not 0<=summary[key]<=1e-5:
            raise ValueError(f'{key} is missing or outside the original acceptance tolerance.')
    if summary['best_bound_keur']>summary['objective_keur']+1e-5:
        raise ValueError('Unrestricted search bound exceeds the exported incumbent.')
    previous=result['final'];results.append(result);summaries.append({'day':day,**summary})
    records.append({'day':day,'status':'PASS','reason':'Native workbook operational audit passed.'})
display(pd.DataFrame(records))
if summaries:
    display(pd.DataFrame(summaries)[['day','objective_eur','relative_gap','search_seconds','polish_seconds','equation_residual']])
    # Saving CSV does not create replacement GDX state or replace archived evidence.
    target=ROOT/'verification/local_reports';target.mkdir(parents=True,exist_ok=True)
    pd.DataFrame(summaries).to_csv(target/'03_native_summaries.csv',index=False)
else:
    print('NOT RUN: no native GAMS outputs. Reader setup succeeded; solver/UC-result checks were not executed.')
# Small source-reader unit checks: these are not native-workbook tests.
assert number('EPS')==0 and number('')==0 and number('2.5')==2.5
try:number(float('inf'))
except ValueError:pass
else:raise AssertionError('A non-finite number was not rejected.')
report={'notebook_completed':True,'native_workbook_audit':'PASS' if results else 'NOT RUN',
        'native_days_audited':len(results),'reader_number_checks':'PASS','days':records,'sode_assessment':'NOT RUN'}
target=ROOT/'verification/local_reports';target.mkdir(parents=True,exist_ok=True)
(target/'03_native_reader.json').write_text(json.dumps(report,indent=2)+'\n')


,day,status,reason
0,1,NOT RUN,No native workbook supplied/generated.
1,2,NOT RUN,No native workbook supplied/generated.
2,3,NOT RUN,No native workbook supplied/generated.
3,4,NOT RUN,No native workbook supplied/generated.
4,5,NOT RUN,No native workbook supplied/generated.
5,6,NOT RUN,No native workbook supplied/generated.
6,7,NOT RUN,No native workbook supplied/generated.


NOT RUN: no native GAMS outputs. Reader setup succeeded; solver/UC-result checks were not executed.


972